# ARC-AGI-3 submission — REPL agent + Qwen3.8-27B-FP8

完整版: vLLM 本地起 Qwen3.8-27B, REPL-agent(模型写代码观察/验证/行动)打游戏。
真提交走网关打隐藏游戏; Save & Run 用公开环境文件小预算验证整条栈。


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen

NOTEBOOK_T0 = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
os.environ["ONLY_RESET_LEVELS"] = "true"
WORKING = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path("out")
WORKING.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("RECORDINGS_DIR", str(WORKING / "recordings"))

import torch
print("GPU:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0) >= (8, 9), "FP8 需要 CC>=8.9, 加速器要选 RTX 6000(NvidiaRtxPro6000)"
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


## 1. 离线装 vLLM + arc-agi


In [ ]:
def find_dir(pattern):
    for p in Path("/kaggle/input").rglob(pattern):
        return p.parent
    raise RuntimeError(f"找不到 {pattern}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links", str(find_dir("vllm-*.whl")), "vllm"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links",
                       "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels", "arc-agi"])
import importlib.metadata as md_
print("vllm", md_.version("vllm"), "| arc-agi", md_.version("arc-agi"))


## 2. 定位源码 bundle 与模型权重


In [ ]:
bundle = next(Path("/kaggle/input").rglob("arc3-jinbo-bundle.json")).parent
sys.path.insert(0, str(bundle))

model_dir = None
for cfg in Path("/kaggle/input").rglob("config.json"):
    if list(cfg.parent.glob("*.safetensors")):
        model_dir = cfg.parent
        break
assert model_dir, "找不到模型目录(要挂 Kaggle Model)"
print("bundle:", bundle, "| model:", model_dir)


## 3. 起 vLLM(权重加载约10-20分钟), 配 agent


In [ ]:
from kaggle_agent.serve_vllm import start_vllm
proc = start_vllm(str(model_dir), port=8000, max_model_len=32768,
                  log_path=str(WORKING / "vllm.log"))
os.environ["A3_LLM_BASE_URL"] = "http://127.0.0.1:8000"
os.environ["A3_LLM_MODEL"] = "local"
os.environ["A3_AGENT"] = "repl"

from kaggle_agent.llm import LLMClient
smoke = LLMClient("http://127.0.0.1:8000", model="local", max_tokens=100)
print("冒烟:", smoke.chat([{"role": "user", "content": "回复OK"}])[:50])


## 4. 真提交: 网关(变量必须硬编码) / 离线: 公开环境文件


In [ ]:
def wait_gateway(base_url, timeout_s=600.0):
    deadline, last = time.monotonic() + timeout_s, ""
    probe = base_url.rstrip("/") + "/api/games"
    while time.monotonic() < deadline:
        try:
            with urlopen(probe, timeout=10) as r:
                if r.status < 500:
                    return
        except Exception as e:
            last = repr(e)
        time.sleep(5)
    raise RuntimeError(f"gateway not ready: {last}")

env_dir = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    os.environ.setdefault("SCHEME", "http")
    os.environ.setdefault("HOST", "gateway")
    os.environ.setdefault("PORT", "8001")
    os.environ.setdefault("OPERATION_MODE", "competition")
    os.environ.setdefault("ENVIRONMENTS_DIR", "")
    wait_gateway(os.environ["ARC_BASE_URL"])
    print("gateway ready")
else:
    cands = [p for p in Path("/kaggle/input").rglob("environment_files") if p.is_dir()] if Path("/kaggle/input").is_dir() else []
    cands += [bundle / "environment_files_sample", Path("environment_files")]
    env_dir = next((str(p) for p in cands if Path(p).is_dir()), None)
    assert env_dir, "找不到离线环境文件目录"
    print("offline env_dir:", env_dir)


## 5. 跑游戏

真提交: 总墙钟 8h 均分给隐藏游戏。离线验证: 2 局小预算把栈跑通即可。


In [ ]:
from kaggle_agent.run_submission import main

if TRUE_SUBMISSION:
    budget = dict(seconds_per_game=600.0, max_actions=200,
                  total_seconds=8 * 3600 - (time.time() - NOTEBOOK_T0))
    games = None
else:
    budget = dict(seconds_per_game=float(os.environ.get("A3_SECONDS_PER_GAME", 240)),
                  max_actions=int(os.environ.get("A3_MAX_ACTIONS", 100)),
                  total_seconds=float(os.environ.get("A3_TOTAL_SECONDS", 600)))
    games = ["ft09", "r11l"]

summary = main(env_dir=env_dir or "environment_files", games=games,
               out_dir=str(WORKING), **budget)


## 6. submission.parquet 门禁 + 结果


In [ ]:
if not TRUE_SUBMISSION:
    import pandas as pd
    pd.DataFrame([["1_0", "1", True, 1]],
                 columns=["row_id", "game_id", "end_of_game", "score"],
                 ).to_parquet(WORKING / "submission.parquet", index=False)
    print("submission.parquet written")
for g in summary["games"]:
    print(f"{g['game_id']:>16} levels {g['levels_completed']}/{g['win_levels']}"
          f" steps={g['steps']} {str(g['state'])[:30]}")
print("llm:", summary.get("llm_stats"))
